# SpaceX Falcon 9 Data Collection using API

This notebook collects historical SpaceX launch data using the SpaceX REST API.

The collected data will be used in the later data wrangling, exploratory data analysis, visualization, and machine learning stages of the capstone project.

## 1. Import Libraries

In [ ]:
import requests
import pandas as pd
import numpy as np

## 2. Request Data from the SpaceX API

The SpaceX API provides historical launch information in JSON format.

In [ ]:
url = "https://api.spacexdata.com/v4/launches/past"

response = requests.get(url)
response.raise_for_status()

launch_data = response.json()

print("Number of launches collected:", len(launch_data))

## 3. Convert API Response to a DataFrame

In [ ]:
df = pd.json_normalize(launch_data)

print("Dataset shape:", df.shape)
df.head()

## 4. Retrieve Rocket Information

The launch records contain rocket IDs. The rocket API is used to identify the corresponding rocket names.

In [ ]:
rockets_url = "https://api.spacexdata.com/v4/rockets"

rockets_response = requests.get(rockets_url)
rockets_response.raise_for_status()

rockets = rockets_response.json()
    

In [ ]:
rocket_lookup = {rocket["id"]: rocket["name"] for rocket in rockets}

df["rocket_name"] = df["rocket"].map(rocket_lookup)

df[["name", "rocket_name", "date_utc", "success"]].head()

## 5. Filter Falcon 9 Launches

The capstone focuses on Falcon 9 launches. Therefore, launches associated with Falcon 9 are selected for further analysis.

In [ ]:
falcon9_df = df[df["rocket_name"].str.contains("Falcon 9", na=False)].copy()

print("Falcon 9 launches:", len(falcon9_df))
falcon9_df.head()

## 6. Select Relevant Launch Information

The collected Falcon 9 launch information will be further processed during the data wrangling stage.

In [ ]:
selected_columns = [
    "name",
    "date_utc",
    "rocket",
    "rocket_name",
    "success",
    "launchpad",
    "payloads",
    "orbit",
    "cores"
]

available_columns = [column for column in selected_columns if column in falcon9_df.columns]

falcon9_data = falcon9_df[available_columns].copy()

falcon9_data.head()

## 7. Save the Collected Data

The Falcon 9 launch data is saved as a CSV file for use in the subsequent data wrangling stage.

In [ ]:
falcon9_data.to_csv("spacex_api_launch_data.csv", index=False)

print("API data collection completed successfully.")